# 03 - Baseline B2: regresión logística regularizada

B2 está cerrado. La configuración seleccionada es `C=0.1` en A, B y C; los reajustes finales en `train` y sus métricas de `val` están registrados en `results/baseline2/`. Este cuaderno presenta esos resultados y deja documentada la demostración de fuga pendiente. No ejecuta búsqueda ni reentrenamiento.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
repo = Path.cwd()
if not (repo / 'results' / 'baseline2').exists():
    repo = Path('/content/repo')
RESULTS_DIR = repo / 'results' / 'baseline2'


## Resultados versionados

Cada fila de resultados se reporta por escenario. La selección usó `val`; `calib` y `test` permanecen reservados. Los modelos serializados no están versionados aquí y su posible copia desde Drive debe verificarse antes de plantear cualquier reentrenamiento.

In [ ]:
# Lee los CSV ya versionados; esta celda no ajusta modelos.
for nombre in ('b2_val_A.csv', 'b2_val_B.csv', 'b2_val_C.csv'):
    ruta = RESULTS_DIR / nombre
    if ruta.exists():
        print(nombre)
        display(pd.read_csv(ruta).round(4))
    else:
        print('No está disponible:', ruta)


## Demostración de fuga: pendiente

La comparación entre partición aleatoria por fila y partición agrupada por `game_id` aún no se ejecutó. Debe medir cuántas partidas aparecen en ambos lados de la partición aleatoria y contrastar las métricas con el protocolo agrupado. No se debe confundir esa demostración con una nueva búsqueda de hiperparámetros.

## Preparación para las tareas pendientes

Usa el mismo runtime de `00_arranque`. Las secciones siguientes se ejecutan de forma independiente.

In [ ]:
# Ejecuta antes 00_arranque en este mismo runtime de Colab.
from pathlib import Path
import sys, json, time, subprocess
from datetime import datetime, timezone
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
REPO_DIR = Path('/content/repo')
DATA_DIR = Path('/content/datos')
assert (REPO_DIR / 'src/ptcg_wp/pending_experiments.py').exists(), 'Actualiza el repo en Colab con estos cambios.'
assert (DATA_DIR / 'particiones.parquet').exists(), 'Ejecuta primero 00_arranque.'
if str(REPO_DIR / 'src') not in sys.path: sys.path.insert(0, str(REPO_DIR / 'src'))
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from ptcg_wp.features import feature_columns
schema = pq.ParquetFile(sorted(DATA_DIR.glob('estados_*.parquet'))[0]).schema.names
FEATURES = feature_columns(schema)
assert len(FEATURES) == 62
SEED = 2026
DRIVE_ROOT = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
dirty = bool(subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip())
RUN_INFO = {'commit': commit, 'working_tree_dirty': dirty, 'seed': SEED, 'run_id_utc': RUN_ID}
print('Commit:', commit, '| cambios locales en Colab:', dirty, '| corrida:', RUN_ID)


## Demostración de fuga

La demostración usa únicamente la configuración finalista `C=0.1`. Con la misma muestra de partidas de `train`, reajusta una regresión logística para cada división: una por fila y otra por `game_id`. Debe reajustar porque cada división tiene partidas de entrenamiento distintas. Usa LBFGS en una muestra pequeña, mientras el B2 final emplea SGD sobre todo `train`; por eso es un diagnóstico de fuga, no una comparación de desempeño entre modelos ni una reutilización del `.joblib` final. Se guardan métricas y solapamientos en Drive. La diferencia observada no garantiza inflación en todas las muestras.

In [ ]:
from ptcg_wp.baselines import ScenarioStream
from ptcg_wp.pending_experiments import leakage_comparison
# Cambia a True solo para ejecutar el diagnóstico.
EJECUTAR_FUGA = False
ESCENARIOS_FUGA = ('A', 'B', 'C')
if EJECUTAR_FUGA:
    for escenario in ESCENARIOS_FUGA:
        out = DRIVE_ROOT / 'b2' / 'demo_fuga' / RUN_ID / f'escenario_{escenario}'
        stream = ScenarioStream(DATA_DIR, escenario, tuple(FEATURES))
        metricas, solapamiento = leakage_comparison(stream, FEATURES, out, seed=SEED, max_games=200, c_value=0.1)
        (out / 'registro.json').write_text(json.dumps(RUN_INFO, indent=2))
        display(metricas); display(solapamiento)
        print('Guardado:', out)


## Recuperar modelos B2 seleccionados

Busca los archivos en Drive y verifica sus metadatos. Los copia a una carpeta de exportación, sin entrenar. Un archivo no encontrado o ambiguo requiere revisión manual.

In [ ]:
import hashlib, shutil, joblib
BUSCAR_MODELOS_B2 = False
if BUSCAR_MODELOS_B2:
    search = DRIVE_ROOT / 'b2'
    dest = DRIVE_ROOT / 'export_repo' / 'baseline2'; dest.mkdir(parents=True, exist_ok=True)
    for escenario in ('A', 'B', 'C'):
        nombre = f'b2_{escenario}_C0.1.joblib'
        encontrados = list(search.rglob(nombre))
        if len(encontrados) != 1:
            print(nombre, ': se esperaba 1 archivo; encontrados:', encontrados); continue
        src = encontrados[0]; item = joblib.load(src)
        assert item.get('escenario') == escenario and float(item['C']) == 0.1
        assert list(item['features']) == list(FEATURES)
        assert item.get('max_epocas') == 600
        target = dest / nombre
        if target.exists():
            print('Ya existe; no se sobrescribe:', target); continue
        shutil.copy2(src, target)
        print('Copiado:', target, '| SHA256:', hashlib.file_digest(target.open('rb'), 'sha256').hexdigest(),
              '| convergencia:', item.get('convergencia'), '| sklearn:', item.get('version_sklearn'))
